# Exercise #5: A bad afternoon with a sloppy teammate's notebook

**The situation.** Your teammate built a Barra-style backtest on the Week 3 data, then left for vacation. The PM meeting is Monday. The notebook runs from top to bottom without a single error, and its headline number is spectacular: a long-short Sharpe ratio near 3.

You remember Week 3. The honest answer was that these eight styles barely forecast returns. Something is wrong.

**Your job.** The notebook contains **six bugs**. None of them crashes. Each one quietly changes a number. Find all six, fix them, and explain each one.

**When everything is fixed, you should get:**

| Number | Target |
|---|---|
| Average stock return in the sample | 0.77% a month |
| Backtest months | 97 |
| Mean IC | 0.012 |
| t-stat of the IC | 0.94 |
| Top-minus-bottom decile spread | 0.19% a month |
| Sharpe ratio (annual) | 0.09 |
| t-stat of the momentum factor return | 1.33 |

**Hint.** Three bugs are in Part 1 (data preparation), one is in Part 2 (the factor-return statistics), and two are in Part 3 (the backtest).

**Using AI.** You may. But what's graded is the **bug log** at the end, where you explain each bug in your own words and predict its effect *before* you fix it. In class, you may be asked to explain any one of your six bugs.

**Due:** Monday, Nov 30, noon. Submit your corrected notebook, with the bug log filled in.

## Part 0. Setup

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.3f}".format)

# The Week 3 class data (week3_data.zip). The link is on the course notes page, not in this public notebook.
DATA_ZIP_LINK = ""     # <- paste the Google Drive link to week3_data.zip between the quotes, then run this cell

import re, zipfile, urllib.request
DATA_DIR = Path("data")
if not (DATA_DIR / "w3_panel.parquet").exists():
    fid = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", DATA_ZIP_LINK)
    assert fid, "Paste the week3_data.zip link from the course notes page into DATA_ZIP_LINK above, then rerun this cell."
    fid = fid.group(1) or fid.group(2)
    urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", "week3_data.zip")
    assert zipfile.is_zipfile("week3_data.zip"), "The download did not return a zip. Check that the link is shared with you, and try again."
    zipfile.ZipFile("week3_data.zip").extractall(".")
    print("downloaded and unzipped week3_data.zip")

STYLES = ["size", "value", "momentum", "beta", "resid_vol", "leverage", "liquidity", "profitability"]
raw = pd.read_parquet(DATA_DIR / "w3_panel.parquet")[["permno", "month", "y", "me_prev", "industry"] + STYLES]
print(f"{len(raw):,} stock-months, {raw.month.nunique()} months, {raw.month.min()} to {raw.month.max()}")
raw.head()

**What the data are (from the Week 3 notes).** Each row is a stock in a month. `y` is the stock's return *in* that month. `me_prev` is its market cap at the end of the *previous* month. The eight Barra styles are measured with information available at the end of the previous month, and are already standardized each month.

## Part 1. Data preparation

*Your teammate's comments are left as they wrote them.*

In [ ]:
p = raw.copy()

# Keep a balanced panel: only stocks with a full history, so every stock is comparable over time.
n_months = p.month.nunique()
p = p[p.groupby("permno").month.transform("nunique") == n_months]

# Lag the styles by one month so we never use information we didn't have yet.
p = p.sort_values(["permno", "month"])
p[STYLES] = p.groupby("permno")[STYLES].shift(1)
p = p.dropna(subset=STYLES)

# Clean the data: returns beyond +/-50% in a month are almost always data errors.
p = p[p["y"].abs() < 0.5]

# Standardize each style across stocks, every month: mean 0, standard deviation 1.
p[STYLES] = p.groupby("month")[STYLES].transform(lambda x: (x - x.mean()) / x.std())

months = sorted(p.month.unique())
print(f"after preparation: {len(p):,} stock-months, {len(months)} months, about {len(p) // len(months):,} stocks a month")
print(f"average stock return in the sample: {100 * p['y'].mean():.2f}% a month")

## Part 2. Barra regression, one month at a time

Each month, regress the cross-section of stock returns on industry dummies and the eight styles, weighted by the square root of last month's market cap (Session 3). The eight style slopes are that month's **style factor returns**.

In [ ]:
rows = {}
for m, g in p.groupby("month"):
    w = np.sqrt(g["me_prev"].values)                       # regression weights: sqrt of market cap
    D = pd.get_dummies(g["industry"]).values.astype(float)  # industry dummies (no constant needed)
    X = np.hstack([D, g[STYLES].values])
    sw = np.sqrt(w)                                         # weighted least squares = OLS on rows scaled by sqrt(weight)
    b = np.linalg.lstsq(X * sw[:, None], g["y"].values * sw, rcond=None)[0]
    rows[m] = b[-len(STYLES):]
f = pd.DataFrame.from_dict(rows, orient="index", columns=STYLES).loc[months]
print(f"{len(f)} months of style factor returns")

In [ ]:
# Is each style's average factor return different from zero?
tstat = f.mean() / f.std() * np.sqrt(12)                # annualized t-stat
pd.DataFrame({"mean, % a month": 100 * f.mean(), "t-stat": tstat})

## Part 3. The backtest

Starting after three years of history, each month: forecast every stock's return as its style exposures times the **average style factor return so far**. Then measure:
- the **IC**: the cross-sectional correlation between the forecast and the realized return;
- the **spread**: the return of the top forecast decile minus the bottom decile, equal-weighted.

In [ ]:
groups = dict(tuple(p.groupby("month")))
ic, spread, dates = [], [], []
for t in range(36, len(months)):
    hist = f.iloc[:t + 1]                                  # use everything we know so far
    g = groups[months[t]]
    pred = g[STYLES].values @ hist.mean().values
    ic.append(np.corrcoef(pred, g["y"].values)[0, 1])
    q = pd.qcut(pred, 10, labels=False)
    spread.append(g["y"].values[q == 9].mean() - g["y"].values[q == 0].mean())
    dates.append(months[t])
ic, spread = pd.Series(ic, index=dates), pd.Series(spread, index=dates)

In [ ]:
ANNUALIZE = np.sqrt(252)                                   # trading days in a year
nav = (1 + spread).cumprod()
results = pd.Series({
    "backtest months": len(spread),
    "mean IC": ic.mean(),
    "t-stat of the IC": ic.mean() / ic.std() * np.sqrt(len(ic)),
    "top-minus-bottom spread, % a month": 100 * spread.mean(),
    "Sharpe ratio (annual)": spread.mean() / spread.std() * ANNUALIZE,
    "max drawdown": (nav / nav.cummax() - 1).min(),
})
results.to_frame("value")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
nav.index = pd.PeriodIndex(nav.index, freq="M").to_timestamp()
nav.plot(ax=ax, title="Top-minus-bottom decile, growth of $1")
plt.tight_layout(); plt.show()

## Bug log

Fill in one row per bug. **Write the "predicted effect" column before you fix the bug.** Then fix it, rerun everything, and record what actually happened.

| # | Cell and line | What's wrong, in your own words | Predicted effect: makes the strategy look better or worse, and why | Sharpe ratio before → after the fix |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |

**Two questions to finish.**
1. Which bug did the most damage? Why is that kind of bug so dangerous in a real research process?
2. Your teammate's notebook made a weak strategy look spectacular. Before you found any bugs, what about its output should have made you suspicious?